# Functional Alignment and GLM Design

**NX-421 – Neural Signals and Signal Processing**

In this notebook, we continue the analysis of the HCP language-task run (`tfMRI_LANGUAGE_LR`), using the outputs of the structural preprocessing (point 1) and of the motion correction and smoothing (points 2a–b).

The notebook covers three steps:

1. **Alignment between anatomical and functional data (2c):** Overlay the fMRI on the T1-weighted image registered to MNI space, assess the quality of the alignment, and discuss which functional preprocessing step can be avoided.
2. **Experimental design matrix (3a):** Build the design matrix used in the GLM analysis of the story and math tasks.
3. **Correlation between regressors (3b):** Compute and plot the correlation matrix of the design matrix regressors.

The design matrix is saved in `outputs/glm` so that it can be used to fit the GLM in the next steps.

## 0. Setup and input data

We use NiBabel to read the NIfTI images, NumPy and Pandas for arrays and tables, Matplotlib and Nilearn for the figures, and Nilearn's `glm` module to build the design matrix (Lab 5).

FSL is loaded to access the MNI152 template and FSLeyes, as in the previous notebooks.

In [1]:
# General imports
import inspect
import os
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt

from nilearn import image, plotting
from nilearn.masking import compute_epi_mask
from nilearn.glm.first_level import make_first_level_design_matrix
from nilearn.plotting import plot_design_matrix

# Load FSL in Neurodesk, as in Lab 2
import module as lmod

await lmod.purge(force=True)
await lmod.load('fsl/6.0.7.18')

# Project directories
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
FUNC_DATA_DIR = DATA_DIR / "fMRI" / "tfMRI_LANGUAGE_LR"
REGISTRATION_DIR = PROJECT_ROOT / "outputs" / "registration_mni"  # point 1
FUNC_PREPROC_DIR = PROJECT_ROOT / "outputs" / "functional"        # points 2a-b
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "glm"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_ROOT)
print("Output directory:", OUTPUT_DIR)

Project directory: /data/Desktop/myfiles/NX421-MiniProject-1
Output directory: /data/Desktop/myfiles/NX421-MiniProject-1/outputs/glm


### 0.1 Input files

We use the following files:

- **From point 1:** the skull-stripped T1w registered to MNI space, with the non-linear ANTs SyN registration (used as the anatomical reference) and with the affine FLIRT registration (for comparison).
- **From points 2a–b:** the motion-corrected fMRI run, its motion parameters (`.par` file from MCFLIRT) and the motion-corrected + smoothed run, on which the GLM will be fitted.
- **From the dataset:** the original run (only its header is read) and the task events.
- **From FSL:** the MNI152 2 mm brain template.

The file names of points 2a–b should be adapted here if they differ.

In [2]:
# Point 1: structural preprocessing
anat_syn_path = REGISTRATION_DIR / "T1w_brain_MNI_SyN.nii.gz"
anat_flirt_path = REGISTRATION_DIR / "T1w_brain_MNI.nii.gz"

# Points 2a-b: functional preprocessing
moco_fmri_path = FUNC_PREPROC_DIR / "tfMRI_LANGUAGE_LR_mcf.nii.gz"
motion_par_path = FUNC_PREPROC_DIR / "tfMRI_LANGUAGE_LR_mcf.par"
smoothed_fmri_path = FUNC_PREPROC_DIR / "tfMRI_LANGUAGE_LR_mcf_smooth.nii.gz"

# Dataset
raw_fmri_path = FUNC_DATA_DIR / "tfMRI_LANGUAGE_LR.nii.gz"
events_path = FUNC_DATA_DIR / "tfMRI_LANGUAGE_LR_task_onsets.csv"

# MNI152 template at the resolution of the fMRI (2 mm)
mni_2mm_path = Path(os.environ["FSLDIR"]) / "data" / "standard" / "MNI152_T1_2mm_brain.nii.gz"

for path in [anat_syn_path, anat_flirt_path, moco_fmri_path, motion_par_path,
             smoothed_fmri_path, raw_fmri_path, events_path, mni_2mm_path]:
    if not path.is_file():
        raise FileNotFoundError(f"Input not found: {path}")
    print(f"[OK] {path.name}")

[OK] T1w_brain_MNI_SyN.nii.gz
[OK] T1w_brain_MNI.nii.gz
[OK] tfMRI_LANGUAGE_LR_mcf.nii.gz
[OK] tfMRI_LANGUAGE_LR_mcf.par
[OK] tfMRI_LANGUAGE_LR_mcf_smooth.nii.gz
[OK] tfMRI_LANGUAGE_LR.nii.gz
[OK] tfMRI_LANGUAGE_LR_task_onsets.csv
[OK] MNI152_T1_2mm_brain.nii.gz


## 1. Alignment Between Anatomical and Functional Data (2c)

To interpret the fMRI results anatomically (and later with atlases), the functional data and the anatomical image must be in the same space. Following Lab 3, we check the alignment by overlaying the two images and looking at:

- the orientation of the functional image,
- the position of the ventricles,
- the match between the brain boundaries of the EPI and of the anatomical image.

### 1.1 Spatial information of the images

We first compare the voxel grids of the fMRI run, the MNI152 2 mm template and the T1w registered to MNI space. The **affine** of an image maps its voxel indices to millimetre coordinates: two images with the same affine and dimensions are sampled on exactly the same grid.

In [3]:
moco_img = nib.load(str(moco_fmri_path))
mni_2mm_img = nib.load(str(mni_2mm_path))
anat_syn_img = nib.load(str(anat_syn_path))

for name, img in [("fMRI (motion-corrected)", moco_img),
                  ("MNI152 2 mm template", mni_2mm_img),
                  ("T1w in MNI space (SyN)", anat_syn_img)]:
    print(name)
    print("  Dimensions:", img.shape)
    print("  Voxel size (mm):", img.header.get_zooms()[:3])
    print("  Orientation:", nib.aff2axcodes(img.affine))
    print("  Affine:\n", np.round(img.affine, 2))

same_grid = moco_img.shape[:3] == mni_2mm_img.shape and np.allclose(moco_img.affine, mni_2mm_img.affine)
print("\nfMRI on the MNI152 2 mm grid:", same_grid)

fMRI (motion-corrected)
  Dimensions: (91, 109, 91, 316)
  Voxel size (mm): (np.float32(2.0), np.float32(2.0), np.float32(2.0))
  Orientation: ('L', 'A', 'S')
  Affine:
 [[  -2.    0.    0.   90.]
 [   0.    2.    0. -126.]
 [   0.    0.    2.  -72.]
 [   0.    0.    0.    1.]]
MNI152 2 mm template
  Dimensions: (91, 109, 91)
  Voxel size (mm): (np.float32(2.0), np.float32(2.0), np.float32(2.0))
  Orientation: ('L', 'A', 'S')
  Affine:
 [[  -2.    0.    0.   90.]
 [   0.    2.    0. -126.]
 [   0.    0.    2.  -72.]
 [   0.    0.    0.    1.]]
T1w in MNI space (SyN)
  Dimensions: (182, 218, 182)
  Voxel size (mm): (np.float32(1.0), np.float32(1.0), np.float32(1.0))
  Orientation: ('L', 'A', 'S')
  Affine:
 [[  -1.   -0.    0.   90.]
 [  -0.    1.   -0. -126.]
 [   0.    0.    1.  -72.]
 [   0.    0.    0.    1.]]

fMRI on the MNI152 2 mm grid: True


The fMRI run has 2 mm isotropic voxels on a 91 × 109 × 91 grid, which is exactly the grid of the MNI152 2 mm template (same dimensions and affine). The HCP functional data are distributed after the HCP "minimal preprocessing" pipeline, which already registers each run non-linearly to MNI space.

The T1w registered in point 1 is also in MNI space, on the 1 mm grid of the template (182 × 218 × 182). The two images therefore share the same millimetre coordinates and can be overlaid directly, without any additional transformation.

### 1.2 Choice of the fMRI volume

For the overlay we use one volume of the **motion-corrected** run (point 2a). We use the unsmoothed run, because smoothing blurs the brain boundaries that we want to compare.

We choose the middle volume of the run: it is far from the first volumes (which have not yet reached the steady state of magnetisation and are brighter) and it is the reference that MCFLIRT uses by default for realignment.

In [ ]:
n_volumes = moco_img.shape[3]
volume_index = n_volumes // 2

fmri_volume = image.index_img(str(moco_fmri_path), volume_index)
nib.save(fmri_volume, OUTPUT_DIR / f"fmri_volume_{volume_index}.nii.gz")

print(f"Selected volume: {volume_index} (t = {volume_index * moco_img.header.get_zooms()[3]:.1f} s)")

### 1.3 Overlay of the fMRI on the anatomical image

We display the fMRI volume (hot colours, semi-transparent) on top of the T1w registered to MNI space with ANTs SyN, which point 1 selected as the anatomical reference. We use the same cut coordinates as in point 1.

In [ ]:
def add_fmri_overlay(display, img):
    # Semi-transparent overlay ("alpha" is called "transparency" in recent Nilearn versions)
    if "transparency" in inspect.signature(display.add_overlay).parameters:
        display.add_overlay(img, cmap="hot", transparency=0.5)
    else:
        display.add_overlay(img, cmap="hot", alpha=0.5)


display = plotting.plot_anat(
    str(anat_syn_path),
    title=f"fMRI volume {volume_index} over the T1w in MNI space (SyN)",
    display_mode="ortho",
    cut_coords=(0, -20, 20),
    dim=-0.3
)
add_fmri_overlay(display, fmri_volume)
plotting.show()

To better see the boundaries, we now do the opposite: the fMRI volume is the background and the edges of the anatomical image are drawn in red. We use one axial section through the lateral ventricles and the insula (z = 10 mm), and compare the two registrations of point 1 (non-linear SyN and affine FLIRT).

In [ ]:
SECTION_Z = 10  # mm

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

disp = plotting.plot_anat(str(anat_syn_path), display_mode="z", cut_coords=[SECTION_Z], axes=axes[0],
                          title="T1w (SyN) + fMRI", dim=-0.3)
add_fmri_overlay(disp, fmri_volume)

for ax, (anat_path, name) in zip(axes[1:], [(anat_syn_path, "SyN"), (anat_flirt_path, "FLIRT")]):
    disp = plotting.plot_anat(fmri_volume, display_mode="z", cut_coords=[SECTION_Z], axes=ax,
                              title=f"fMRI + T1w edges ({name})")
    disp.add_edges(str(anat_path), color="r")

fig.suptitle(f"Functional-anatomical alignment, volume {volume_index}, axial section z = {SECTION_Z} mm")
plt.show()

### 1.4 Quantitative check of the alignment

To complement the visual inspection, we compare the brain masks of the two modalities on the fMRI grid, with the Dice coefficient $D = 2|A \cap B| / (|A| + |B|)$ (1 for a perfect overlap). The functional brain mask is computed from the selected volume, and the anatomical masks are the brain voxels of the skull-stripped T1w in MNI space, resampled to the fMRI grid. The MNI152 template mask is used as a reference.

In [ ]:
def dice(mask_a, mask_b):
    return 2 * np.logical_and(mask_a, mask_b).sum() / (mask_a.sum() + mask_b.sum())


fmri_mask = compute_epi_mask(fmri_volume).get_fdata() > 0

anatomical_masks = {
    "T1w in MNI (SyN)": anat_syn_path,
    "T1w in MNI (FLIRT)": anat_flirt_path,
    "MNI152 2 mm template": mni_2mm_path,
}

for name, path in anatomical_masks.items():
    # Small threshold to ignore near-zero values left by interpolation outside the brain
    brain = image.math_img("img > 0.01 * np.max(img)", img=str(path))
    mask = image.resample_to_img(brain, fmri_volume, interpolation="nearest").get_fdata() > 0
    print(f"Dice (fMRI brain mask vs {name}): {dice(fmri_mask, mask):.3f}")

### 1.5 Interactive inspection with FSLeyes

As in Lab 3, we also load the anatomical image and the motion-corrected run in FSLeyes. Changing the opacity of the fMRI and navigating through the slices (and volumes) allows to check the alignment in the whole brain.

In [ ]:
subprocess.Popen([
    "fsleyes",
    str(anat_syn_path),
    str(moco_fmri_path), "--cmap", "hot", "--alpha", "50"
])

print("T1w (SyN) and motion-corrected fMRI loaded in FSLeyes.")

### 1.6 Alignment Results

The fMRI volume and the T1w in MNI space are well aligned:

- **Orientation.** The functional image has the same (LAS) orientation as the anatomy and the template, and left/right, anterior/posterior and superior/inferior directions match in the three views.
- **Ventricles.** The lateral ventricles, which appear dark in the EPI, fall on the ventricles of the T1w.
- **Brain boundaries.** The outer boundary of the EPI follows the edges of the anatomical brain. The midline (interhemispheric fissure) is at the same position in both images.

The non-linear (SyN) T1w is the better anatomical reference for this check. The HCP pipeline registered the functional data to MNI space with a non-linear warp, which an affine transformation (FLIRT) cannot reproduce: with FLIRT, the cortical edges can be shifted locally with respect to the EPI. Point 1 already found that the SyN registration is closer to the MNI template than the FLIRT registration (correlation 0.777 vs 0.625), and the Dice coefficients printed above quantify the same difference with respect to the fMRI.

Some mismatches remain, as expected:

- **Resolution.** The EPI has a much lower resolution (2 mm instead of 0.7 mm originally), so the fine cortical folds cannot be matched.
- **Signal dropout.** Near air–tissue interfaces (orbitofrontal cortex and inferior temporal lobes) the EPI signal drops because of magnetic susceptibility artefacts. In these regions the EPI looks "smaller" than the anatomy, which is a property of the acquisition and not a registration error.

**Which functional preprocessing step can be avoided?** The **registration of the fMRI to the anatomical image and to MNI space** (coregistration with `epi_reg` and normalisation, Lab 3). The plots show that the functional volume already overlaps the anatomy in MNI space without applying any transformation: its voxel grid is exactly the MNI152 2 mm grid, and the brain boundaries and ventricles match those of the T1w registered to MNI. Computing a new EPI → T1 → MNI transformation would not improve the alignment and would only add an interpolation step that blurs the data.

## 2. Experimental Design Matrix (3a)

The goal of the GLM is to investigate the activation of the brain during the **math** and **story** tasks. The design matrix describes, for each volume of the run, the expected signal of each modelled process.

### 2.1 Functional run used for the GLM

The GLM will be fitted on the motion-corrected and smoothed run of point 2b. Its number of volumes and repetition time (TR) define the time axis of the design matrix.

In [ ]:
fmri_img = nib.load(str(smoothed_fmri_path))

n_scans = fmri_img.shape[3]
t_r = float(fmri_img.header.get_zooms()[3])

# If volumes were removed at the start of the run in point 2a, the time axis starts later
n_removed = nib.load(str(raw_fmri_path)).shape[3] - n_scans

# Time of each volume (HCP uses multiband acquisition, so no slice-timing offset is applied)
frame_times = (np.arange(n_scans) + n_removed) * t_r

print("Number of volumes:", n_scans)
print("Volumes removed at the start of the run:", n_removed)
print(f"TR: {t_r:.2f} s")
print(f"Duration: {n_scans * t_r:.1f} s")

### 2.2 Task events

The timing of the task is given in `tfMRI_LANGUAGE_LR_task_onsets.csv`, with one row per event (onset and duration in seconds, and event type).

In [ ]:
events_all = pd.read_csv(events_path)

print("Columns:", list(events_all.columns))
print(f"Number of events: {len(events_all)}\n")

# Number of events and durations for each event type
events_summary = (
    events_all.groupby("trial_type")["duration"]
    .agg(n_events="count", mean_duration_s="mean", total_duration_s="sum")
    .round(2)
)
events_summary

The file contains two levels of description:

- **Block events:** `story` (4 blocks of 26–29 s) and `math` (9 blocks of about 12 s, each containing two to three arithmetic trials).
- **Sub-events nested inside the blocks:** the presentation of the stimulus (`present_story`, `present_math`), the question (`question_story`, `question_math`) and the response period (`response_story`, `response_math`).

In addition, short `cue` events (about 0.8 s) announce the beginning of each new block.

The timeline below shows how these events are organised during the run.

In [ ]:
lanes = ["story", "present_story", "question_story", "response_story", "cue",
         "math", "present_math", "question_math", "response_math"]
colors = {"story": "tab:red", "math": "tab:blue", "cue": "tab:gray"}

fig, ax = plt.subplots(figsize=(14, 4))

for i, trial_type in enumerate(lanes):
    rows = events_all[events_all["trial_type"] == trial_type]
    color = colors.get(trial_type, "tab:red" if "story" in trial_type else "tab:blue")
    alpha = 1.0 if trial_type in colors else 0.5
    ax.broken_barh(list(zip(rows["onset"], rows["duration"])), (i - 0.4, 0.8),
                   facecolors=color, alpha=alpha)

ax.axvline(frame_times[-1] + t_r, color="black", linestyle="--", label="End of acquisition")
ax.set_yticks(range(len(lanes)))
ax.set_yticklabels(lanes)
ax.invert_yaxis()
ax.set_xlabel("Time (s)")
ax.set_title("Task events of the LANGUAGE_LR run")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

Story and math blocks alternate during the whole run, separated only by the short cues. The last math block continues after the end of the acquisition (dashed line): it is truncated in the design matrix, which only covers the acquired volumes.

### 2.3 Choice of the task regressors

We model the task at the **block level**, with one regressor for `story` and one for `math`. Block regressors capture the sustained activity during each task and give the most statistical power in a block design.

We also include the `cue` events as a third task regressor. The cues are a separate process (preparing for the next block), and modelling them prevents this activity from being absorbed into the story/math estimates.

We do **not** add the sub-events (`present_*`, `question_*`, `response_*`) to this model. They are nested inside the blocks and together cover the same time intervals, so their regressors would almost be a linear combination of the block regressors. The design matrix would then be nearly rank-deficient and the betas could not be estimated reliably.

All the time not covered by these events acts as the **implicit baseline** of the model. We measure how much baseline there is in this run.

In [ ]:
TASK_CONDITIONS = ["story", "math", "cue"]

events = events_all[events_all["trial_type"].isin(TASK_CONDITIONS)]
events = events[["onset", "duration", "trial_type"]].sort_values("onset").reset_index(drop=True)
events.to_csv(OUTPUT_DIR / "glm_events.csv", index=False)

# Time of the run not covered by any modelled event (implicit baseline), on a 10 ms grid
fine_times = np.arange(frame_times[0], frame_times[-1] + t_r, 0.01)
covered = np.zeros(fine_times.size, dtype=bool)

for onset, duration in zip(events["onset"], events["duration"]):
    covered |= (fine_times >= onset) & (fine_times < onset + duration)

baseline_s = (~covered).sum() * 0.01

print("Modelled events:", events["trial_type"].value_counts().to_dict())
print(f"Time covered by story/math/cue: {covered.mean() * 100:.1f} % of the run")
print(f"Implicit baseline: {baseline_s:.1f} s out of {fine_times.size * 0.01:.1f} s")

Almost the entire run is covered by the task events: there are only a few seconds of implicit baseline, spread over the short gaps between the cues and the blocks. This is an important property of this paradigm: **math is the control condition of story** (and vice versa), and there is no real rest period. We will see its consequences on the correlations between regressors (3b).

### 2.4 Confound regressors

Following Lab 5, the design matrix also contains confound (no-interest) regressors:

- **The 6 rigid-body motion parameters** (3 rotations, 3 translations) estimated by MCFLIRT in point 2a, and their **temporal derivatives**. They model the signal changes caused by head motion that remain after realignment.
- **One spike regressor per outlier volume.** Each spike regressor is 1 at one volume and 0 elsewhere, which censors that volume from the estimation without removing it from the time series (the timing of the design is unchanged). We use them for:
  - the volumes with a framewise displacement (FD) above **0.5 mm**, with FD computed from the motion parameters as in Lab 3 (Power et al. 2012);
  - the **non-steady-state volumes** at the start of the run (the first two volumes, whose global signal is about 1 % and 0.5 % higher than in the rest of the run), if they were not already removed in point 2a. They are especially important here: at the start of the run the task regressors are still close to zero (the HRF takes a few seconds to rise), so these bright volumes would make up a large part of the implicit baseline and bias all the task betas.

In [ ]:
FD_THRESHOLD = 0.5            # mm
NONSTEADY_VOLUMES = [0, 1]    # first volumes of the original run

MOTION_COLUMNS = ["Rotation x", "Rotation y", "Rotation z",
                  "Translation x", "Translation y", "Translation z"]


def load_mot_params_fsl_6_dof(path):
    # MCFLIRT .par file: 3 rotations (rad) followed by 3 translations (mm), one row per volume
    return pd.DataFrame(np.loadtxt(path), columns=MOTION_COLUMNS)


def compute_FD_power(mot_params):
    # Framewise displacement (Power et al. 2012), as in Lab 3
    framewise_diff = mot_params.diff().iloc[1:]
    # Rotations converted to displacements on a sphere of 50 mm radius
    converted_rots = framewise_diff[["Rotation x", "Rotation y", "Rotation z"]] * 50
    trans_params = framewise_diff[["Translation x", "Translation y", "Translation z"]]
    return converted_rots.abs().sum(axis=1) + trans_params.abs().sum(axis=1)


mot_params = load_mot_params_fsl_6_dof(motion_par_path)

if len(mot_params) != n_scans:
    raise ValueError("The motion parameters do not match the number of volumes of the run.")

# FD of the first volume is set to 0
fd = np.concatenate([[0.0], compute_FD_power(mot_params).to_numpy()])
fd_outliers = np.where(fd > FD_THRESHOLD)[0].tolist()

# Non-steady-state volumes still present in the run (indices in the current run)
nonsteady = [v - n_removed for v in NONSTEADY_VOLUMES if v >= n_removed]

print(f"Mean FD: {fd.mean():.3f} mm, max FD: {fd.max():.3f} mm")
print(f"Volumes with FD > {FD_THRESHOLD} mm: {fd_outliers}")
print(f"Non-steady-state volumes still in the run: {nonsteady}")

### 2.5 Building the design matrix

The design matrix is built with `make_first_level_design_matrix`, as in Lab 5:

- **Task regressors:** each event is modelled as a boxcar (1 during the event, 0 otherwise) and **convolved with the canonical SPM haemodynamic response function (HRF)**, since the BOLD signal is a delayed and smoothed version of the neural activity.
- **Drift regressors:** a discrete cosine basis with a high-pass cut-off of 1/128 Hz models slow scanner drifts. A story + math cycle lasts about 40–55 s, well below the 128 s cut-off, so the task signal is not removed by the filter.
- **Confound regressors:** the motion parameters, their derivatives and the spike regressors defined above.
- **Constant:** the mean signal of each voxel.

In [ ]:
HRF_MODEL = "spm"
HIGH_PASS = 1 / 128  # Hz

# Confound regressors: motion parameters (demeaned), their derivatives and spike regressors
confounds = pd.DataFrame(index=range(n_scans))

motion = mot_params.to_numpy()
motion_names = ["rot_x", "rot_y", "rot_z", "trans_x", "trans_y", "trans_z"]

for i, name in enumerate(motion_names):
    confounds[name] = motion[:, i] - motion[:, i].mean()
for i, name in enumerate(motion_names):
    confounds[name + "_deriv"] = np.concatenate([[0.0], np.diff(motion[:, i])])

spike_volumes = [(v, "nonsteady") for v in nonsteady]
spike_volumes += [(v, "fd_outlier") for v in fd_outliers if v not in nonsteady]

for volume, kind in spike_volumes:
    spike = np.zeros(n_scans)
    spike[volume] = 1.0
    confounds[f"{kind}_{volume}"] = spike

design_matrix = make_first_level_design_matrix(
    frame_times,
    events,
    hrf_model=HRF_MODEL,
    drift_model="cosine",
    high_pass=HIGH_PASS,
    add_regs=confounds.to_numpy(),
    add_reg_names=list(confounds.columns),
)

design_matrix.to_csv(OUTPUT_DIR / "design_matrix.csv")

print("Design matrix shape (volumes x regressors):", design_matrix.shape)
print("Regressors:", list(design_matrix.columns))
print("Rank:", np.linalg.matrix_rank(design_matrix.to_numpy()))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
plot_design_matrix(design_matrix,ax= ax)
ax.set_title("Design matrix")
plt.tight_layout()
plt.show()

To better see the effect of the HRF convolution, we plot the three task regressors over time, together with the timing of the events.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3.5))

for condition, color in [("story", "tab:red"), ("math", "tab:blue"), ("cue", "tab:gray")]:
    rows = events[events["trial_type"] == condition]
    for onset, duration in zip(rows["onset"], rows["duration"]):
        ax.axvspan(onset, onset + duration, color=color, alpha=0.12, lw=0)
    ax.plot(frame_times, design_matrix[condition], color=color, label=condition)

ax.set_xlim(frame_times[0], frame_times[-1] + t_r)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Regressor value")
ax.set_title("Task regressors after convolution with the HRF (shaded: event timing)")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

### 2.6 Design Matrix Results

The design matrix has one row per volume and the following columns:

- **3 task regressors** (`cue`, `math`, `story`), ordered alphabetically by Nilearn.
- **12 motion regressors:** the 6 motion parameters and their derivatives.
- **Spike regressors** for the non-steady-state volumes and the high-motion volumes (if any).
- **3 cosine drift regressors**, given by the 1/128 Hz cut-off and the 227 s duration of the run.
- **The constant.**

The matrix has full rank, so all the betas can be estimated.

The plot of the task regressors shows the effect of the HRF convolution. Each block regressor rises about 5 s after the block onset, reaches a plateau with a small overshoot, and returns to zero with a small undershoot after the block. The story and math regressors alternate almost perfectly: when one is high, the other is close to zero. The cue regressor has a much smaller amplitude because the cues are short (0.8 s), and it peaks at the transitions between blocks. The last math regressor is cut at the end of the acquisition.

## 3. Correlation Between Regressors (3b)

We compute the Pearson correlation between every pair of regressors of the design matrix. The constant is excluded because it has zero variance.

Highly correlated regressors share variance: the GLM cannot attribute this shared variance to one regressor or the other, which inflates the variance of their betas and reduces the statistical power (collinearity).

In [ ]:
regressors = design_matrix.drop(columns="constant")
corr_matrix = regressors.corr()
corr_matrix.to_csv(OUTPUT_DIR / "regressor_correlations.csv")

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr_matrix, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_matrix)))
ax.set_yticks(range(len(corr_matrix)))
ax.set_xticklabels(corr_matrix.columns, rotation=90)
ax.set_yticklabels(corr_matrix.columns)
fig.colorbar(im, ax=ax, label="Pearson correlation")
ax.set_title("Correlation matrix of the design matrix regressors")
plt.tight_layout()
plt.show()

We zoom on the task regressors and the 6 motion parameters, and print the values.

In [ ]:
zoom_columns = TASK_CONDITIONS + motion_names
zoom = corr_matrix.loc[zoom_columns, zoom_columns]

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(zoom, cmap="RdBu_r", vmin=-1, vmax=1)
for i in range(len(zoom)):
    for j in range(len(zoom)):
        ax.text(j, i, f"{zoom.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_xticks(range(len(zoom)))
ax.set_yticks(range(len(zoom)))
ax.set_xticklabels(zoom.columns, rotation=45, ha="right")
ax.set_yticklabels(zoom.columns)
fig.colorbar(im, ax=ax, label="Pearson correlation")
ax.set_title("Task and motion regressors")
plt.tight_layout()
plt.show()

motion_regressors = [c for c in corr_matrix.columns if c.startswith(("rot_", "trans_"))]
max_motion_corr = corr_matrix.loc[TASK_CONDITIONS, motion_regressors].abs().to_numpy().max()
print(f"Maximum |correlation| between task and motion regressors: {max_motion_corr:.2f}\n")

zoom.round(2)

The correlation matrix only describes pairs of regressors. To quantify how well each task regressor can be estimated given **all** the other regressors (including the constant), we also compute its **variance inflation factor**: VIF$_j = 1 / (1 - R^2_j)$, where $R^2_j$ is obtained by regressing regressor $j$ on all the other columns. A VIF of 1 means that the regressor is orthogonal to the rest of the model; values above 10 are usually considered problematic.

We also compute it for the difference story − math, which is the quantity tested by a story vs math contrast.

In [ ]:
def variance_inflation_factor(design, column):
    # Regress one column on all the other columns of the design matrix
    y = design[column].to_numpy()
    X = design.drop(columns=column).to_numpy()
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    residuals = y - X @ beta
    r2 = 1 - residuals.var() / y.var()
    return 1 / (1 - r2)


vif = pd.Series({c: variance_inflation_factor(design_matrix, c) for c in TASK_CONDITIONS}, name="VIF")

# Same design, with story and math replaced by their difference and their sum
design_diff = design_matrix.copy()
design_diff["story_minus_math"] = design_matrix["story"] - design_matrix["math"]
design_diff["story_plus_math"] = design_matrix["story"] + design_matrix["math"]
design_diff = design_diff.drop(columns=["story", "math"])

print(vif.round(2).to_string())
print(f"\nVIF of (story - math): {variance_inflation_factor(design_diff, 'story_minus_math'):.2f}")
print(f"VIF of (story + math): {variance_inflation_factor(design_diff, 'story_plus_math'):.2f}")

### 3.1 Regressor Correlation Results

**Story and math (r ≈ −0.94).** These two regressors are strongly anti-correlated, because the blocks alternate during the whole run with almost no rest (about 97 % of the run is covered by story, math or cue events). Their sum is therefore nearly constant, i.e. nearly collinear with the constant regressor. This is confirmed by the VIF of story and math (printed above), well above the usual limit of 10: even with only the task, drift and constant regressors, it is already about 18. In practice, the betas of story and math taken individually (each compared with the implicit baseline) will be estimated with a large variance, and their individual statistical maps will have little power.

**Story − math (VIF close to 1).** The difference between the two regressors is almost orthogonal to the rest of the model, so a story vs math contrast is estimated very efficiently. This is how the task was designed: math is the active control condition of story (Binder et al. 2011), not a rest period. The GLM analysis should therefore focus on contrasts between the two tasks.

**Cue (r ≈ −0.1 with story and math, VIF about 2).** The cue regressor is only weakly correlated with the block regressors because the cues fall in the short transitions between blocks. It can be estimated separately, although it has little power since it only contains 7 short events.

**Drift regressors.** They are only weakly correlated with the task regressors (|r| ≤ 0.2). The block alternation has a period of about 40–55 s, much faster than the 128 s cut-off, so the high-pass filter removes very little task signal.

**Motion regressors.** Their correlation with the task regressors (maximum printed above) tells whether the head motion is locked to the task. A low correlation means that the motion regressors remove motion-related variance without removing task-related signal. A high correlation would indicate task-correlated motion (e.g. during the button presses): in that case, part of the task effect would be absorbed by the motion regressors and the corresponding betas would be underestimated. The motion parameters can also be correlated with each other (e.g. rotations and translations along related axes), and with the cosine drift regressors when the head position drifts slowly during the run. This is not a problem for the task effects, since these are nuisance regressors whose betas are not interpreted.

**Spike regressors.** They are almost uncorrelated with the task and drift regressors, since each one contains a single non-zero volume. A spike regressor of a high-motion volume is, however, correlated with the motion derivatives, which are large precisely at that volume.

## 4. Summary

1. **Functional-anatomical alignment (2c):** The HCP fMRI run is already in MNI152 space (same 2 mm grid as the template), and overlays well with the T1w registered to MNI in point 1, especially with the non-linear SyN registration. The registration of the fMRI to the anatomy and to MNI space can therefore be skipped.
2. **Design matrix (3a):** Block regressors for story and math plus a cue regressor, convolved with the SPM HRF. Confounds: motion parameters and their derivatives, spike regressors for the non-steady-state and high-motion volumes, and cosine drifts (1/128 Hz).
3. **Regressor correlations (3b):** Story and math are strongly anti-correlated because the task has almost no rest, so their individual effects are poorly estimated. Their difference, however, is almost orthogonal to the rest of the model, so the story vs math contrast is well estimated.

The design matrix (`design_matrix.csv`), the events used (`glm_events.csv`) and the correlation matrix are saved in `outputs/glm`. The GLM can be fitted directly with this design matrix:

```python
from nilearn.glm.first_level import FirstLevelModel

design_matrix = pd.read_csv(OUTPUT_DIR / "design_matrix.csv", index_col=0)
fmri_glm = FirstLevelModel(noise_model="ar1", standardize=False)
fmri_glm = fmri_glm.fit(str(smoothed_fmri_path), design_matrices=[design_matrix])
```

In [ ]:
# Final output checks
print("Alignment and GLM design - Final Outputs")
print("-" * 60)

for path in sorted(OUTPUT_DIR.glob("*")):
    if path.is_file():
        print(f"[OK] {path.relative_to(PROJECT_ROOT)}")